## 🔹 Task 1: Conceptual Questions


# Why can’t we just use the delta rule for learning weights of hidden layers?

The delta rule cannot directly train hidden layers because hidden neurons do not have a known target output or direct error value. In the output layer, the network can easily calculate the error by comparing the predicted output with the correct answer. For example, if the correct output is 1 and the network predicts 0.2, the error can be directly calculated. This error is then used to update the weights using the delta rule.

However, hidden neurons do not produce the final answer. They only pass information to the next layer and help the network reach the final prediction. Because of this, we do not know what the “correct” output of a hidden neuron should be. Therefore, hidden layers cannot directly calculate an error term like (t−y), which is required in the delta rule.

To solve this problem, neural networks use backpropagation. Backpropagation takes the error from the output layer and sends it backward through the network. In this way, the network can estimate how much each hidden neuron contributed to the final error and update the hidden layer weights accordingly. This allows deep neural networks with multiple hidden layers to learn effectively.


# How far is training neural networks non-deterministic? How does randomness influence training speed and/or resulting model performance?

Training neural networks is partially non-deterministic because the training process contains several sources of randomness. This means that even if we use the same dataset, same model architecture, and same training code, the final model may still produce slightly different results each time it is trained.

One major source of randomness is random weight initialization. Before training begins, the network weights are assigned random values. Since training starts from different initial points, the network may follow different learning paths and reach different solutions. Another source of randomness is the random shuffling of training data and mini-batches. Neural networks learn step by step, so changing the order of the data changes the sequence of weight updates, which can affect the final model.

Techniques such as dropout also add randomness by randomly disabling some neurons during training. This forces the network to learn more robust features and helps reduce overfitting. In addition, GPU computations and floating-point operations can sometimes create very small numerical differences that grow during training.

Randomness affects both training speed and model performance. Good random initialization can help the network converge faster and achieve better accuracy, while poor initialization may slow down learning or lead to weaker results. Similarly, different mini-batch orders may lead the model toward different local minima. Sometimes randomness helps the model escape poor solutions and improves generalization performance.

Therefore, neural network training is not completely deterministic. Small random differences during training can lead to different learning behaviors, training times, and final accuracies, even when the same model is trained multiple times.


# Exercise 8 — Multi-Layer Perceptrons: Backpropagation

---

## Task 1: Conceptual Questions

### a) Why can't we just use the delta rule for learning weights of hidden layers?

The **delta rule** updates a weight using the formula

$$\Delta w_{ij} = \eta \cdot (o_i - y_i) \cdot f'(net_i) \cdot o_j$$

This requires the **desired (target) output** $o_i$ for the neuron whose weight is being updated. For **output neurons**, this target is given directly by the training data (the label). For **hidden neurons**, however, there is no such target — the training data only tells us what the _final_ output should look like, not what each hidden neuron's activation "should" be.

Therefore, the error of a hidden neuron cannot be computed directly as $(o_i - y_i)$. Instead, its error has to be **derived indirectly**, by propagating the errors of the neurons in the _next_ layer backward through the weighted connections:

$$\delta_i^{(l)} = f'(net_i)\sum_k \delta_k^{(l+1)} w_{ki}^{(l+1)}$$

This is exactly the idea of **backpropagation**: the error signal is passed backward, layer by layer, weighted by the connection strengths, so every hidden neuron gets a proportional "share of the blame" for the final output error.

### b) How far is training neural networks non-deterministic?

Neural network training is **inherently non-deterministic** for several reasons:

- **Random weight initialization** — different starting points lead to different trajectories through the loss landscape.
- **Random data shuffling / mini-batch order** — changes the sequence of gradient updates.
- **Stochastic optimizers** (SGD, Adam, dropout, data augmentation) — introduce randomness at every step.
- **Hardware-level non-determinism** — parallel floating-point summation on GPUs can produce slightly different results between runs even with a fixed seed.

**Effect on speed:** A poor (e.g., unlucky) initialization can slow convergence dramatically or cause the network to get stuck in flat regions, saddle points, or poor local minima. Good initialization schemes (Xavier/He) are designed to keep gradients well-scaled and speed up convergence.

**Effect on performance:** Because the loss surface of a neural network is non-convex, different random seeds can lead to convergence at **different local minima**, which can have different generalization performance. This is why results are often averaged over multiple runs/seeds in research, and why fixing a random seed only gives _reproducibility_, not _determinism_ in a deeper sense (the underlying process is still stochastic).

---

## Task 2: Batch Gradient Descent

### Network Setup

**Hidden layer weights (Layer 1)** — convention $w_{ij}^{(1)}$: weight from input neuron $j$ into hidden neuron $i$ ($j=3$ is the bias $b_1=1$):

|            | from $x_1$        | from $x_2$        | from $b_1=1$      |
| ---------- | ----------------- | ----------------- | ----------------- |
| into $h_1$ | $w_{11}^{(1)}=1$  | $w_{12}^{(1)}=-2$ | $w_{13}^{(1)}=-1$ |
| into $h_2$ | $w_{21}^{(1)}=-1$ | $w_{22}^{(1)}=2$  | $w_{23}^{(1)}=1$  |

**Output layer weights (Layer 2)** — convention $w_{ij}^{(2)}$: weight from hidden neuron $j$ into output neuron $i$ ($j=3$ is the bias $b_2=1$):

|            | from $h_1$       | from $h_2$        | from $b_2=1$      |
| ---------- | ---------------- | ----------------- | ----------------- |
| into $y_1$ | $w_{11}^{(2)}=2$ | $w_{12}^{(2)}=-1$ | $w_{13}^{(2)}=1$  |
| into $y_2$ | $w_{21}^{(2)}=1$ | $w_{22}^{(2)}=2$  | $w_{23}^{(2)}=-1$ |

**Net input equations:**

$$net_{h1} = w_{11}^{(1)}x_1 + w_{12}^{(1)}x_2 + w_{13}^{(1)}\cdot 1 \qquad net_{h2} = w_{21}^{(1)}x_1 + w_{22}^{(1)}x_2 + w_{23}^{(1)}\cdot 1$$

$$net_{y1} = w_{11}^{(2)}h_1 + w_{12}^{(2)}h_2 + w_{13}^{(2)}\cdot 1 \qquad net_{y2} = w_{21}^{(2)}h_1 + w_{22}^{(2)}h_2 + w_{23}^{(2)}\cdot 1$$

All activations use $y = \sigma(net)$, all values rounded to 2 decimals at every step, per the assignment instructions.

---

### Step 1 — Forward Pass (all 3 samples)

#### Sample $S^{(1)}$: $X=[1,1]$, Target $Y=[1,0]$

$$net_{h1} = 1(1) + (-2)(1) + (-1)(1) = -2 \;\rightarrow\; h_1=\sigma(-2)=0.12$$
$$net_{h2} = -1(1) + 2(1) + 1(1) = 2 \;\rightarrow\; h_2=\sigma(2)=0.88$$

$$net_{y1} = 2(0.12) - 1(0.88) + 1(1) = 0.36 \;\rightarrow\; y_1=\sigma(0.36)=0.59$$
$$net_{y2} = 1(0.12) + 2(0.88) - 1(1) = 0.88 \;\rightarrow\; y_2=\sigma(0.88)=0.71$$

#### Sample $S^{(2)}$: $X=[3,1]$, Target $Y=[0,1]$

$$net_{h1} = 1(3) + (-2)(1) + (-1)(1) = 0 \;\rightarrow\; h_1=\sigma(0)=0.50$$
$$net_{h2} = -1(3) + 2(1) + 1(1) = 0 \;\rightarrow\; h_2=\sigma(0)=0.50$$

$$net_{y1} = 2(0.50) - 1(0.50) + 1(1) = 1.50 \;\rightarrow\; y_1=\sigma(1.50)=0.82$$
$$net_{y2} = 1(0.50) + 2(0.50) - 1(1) = 0.50 \;\rightarrow\; y_2=\sigma(0.50)=0.62$$

#### Sample $S^{(3)}$: $X=[0,0]$, Target $Y=[0.5,0.5]$

$$net_{h1} = 1(0) + (-2)(0) + (-1)(1) = -1 \;\rightarrow\; h_1=\sigma(-1)=0.27$$
$$net_{h2} = -1(0) + 2(0) + 1(1) = 1 \;\rightarrow\; h_2=\sigma(1)=0.73$$

$$net_{y1} = 2(0.27) - 1(0.73) + 1(1) = 0.81 \;\rightarrow\; y_1=\sigma(0.81)=0.69$$
$$net_{y2} = 1(0.27) + 2(0.73) - 1(1) = 0.73 \;\rightarrow\; y_2=\sigma(0.73)=0.67$$

#### Forward Pass Summary

| Sample    | $net_{h1}$ | $h_1$ | $net_{h2}$ | $h_2$ | $net_{y1}$ | $y_1$ | $net_{y2}$ | $y_2$ |
| --------- | ---------- | ----- | ---------- | ----- | ---------- | ----- | ---------- | ----- |
| $S^{(1)}$ | -2.00      | 0.12  | 2.00       | 0.88  | 0.36       | 0.59  | 0.88       | 0.71  |
| $S^{(2)}$ | 0.00       | 0.50  | 0.00       | 0.50  | 1.50       | 0.82  | 0.50       | 0.62  |
| $S^{(3)}$ | -1.00      | 0.27  | 1.00       | 0.73  | 0.81       | 0.69  | 0.73       | 0.67  |

---

### Step 2 — Backward Pass: Local Errors $\delta$

**Output layer:** $\delta_i^{(2)} = f'(net_i)\,(o_i-y_i)$

**Hidden layer:** $\delta_i^{(1)} = f'(net_i)\sum_k \delta_k^{(2)} w_{ki}^{(2)}$

For $h_1$: $\;\sum_k \delta_k^{(2)}w_{ki} = \delta_1^{(2)}w_{11}^{(2)} + \delta_2^{(2)}w_{21}^{(2)}$
For $h_2$: $\;\sum_k \delta_k^{(2)}w_{ki} = \delta_1^{(2)}w_{12}^{(2)} + \delta_2^{(2)}w_{22}^{(2)}$

#### Sample $S^{(1)}$ ($o=[1,0]$)

$$\delta_1^{(2)} = \sigma'(0.36)\,(1-0.59) = 0.24 \times 0.41 = 0.10$$
$$\delta_2^{(2)} = \sigma'(0.88)\,(0-0.71) = 0.21 \times (-0.71) = -0.15$$

$$\delta_1^{(1)} = \sigma'(-2.00)\big[\delta_1^{(2)}(2) + \delta_2^{(2)}(1)\big] = 0.11\big[0.10(2)+(-0.15)(1)\big] = 0.11(0.05) = 0.01$$
$$\delta_2^{(1)} = \sigma'(2.00)\big[\delta_1^{(2)}(-1) + \delta_2^{(2)}(2)\big] = 0.11\big[0.10(-1)+(-0.15)(2)\big] = 0.11(-0.40) = -0.04$$

#### Sample $S^{(2)}$ ($o=[0,1]$)

$$\delta_1^{(2)} = \sigma'(1.50)\,(0-0.82) = 0.15 \times (-0.82) = -0.12$$
$$\delta_2^{(2)} = \sigma'(0.50)\,(1-0.62) = 0.24 \times 0.38 = 0.09$$

$$\delta_1^{(1)} = \sigma'(0.00)\big[(-0.12)(2)+(0.09)(1)\big] = 0.25(-0.15) = -0.04$$
$$\delta_2^{(1)} = \sigma'(0.00)\big[(-0.12)(-1)+(0.09)(2)\big] = 0.25(0.30) = 0.08$$

#### Sample $S^{(3)}$ ($o=[0.5,0.5]$)

$$\delta_1^{(2)} = \sigma'(0.81)\,(0.5-0.69) = 0.21 \times (-0.19) = -0.04$$
$$\delta_2^{(2)} = \sigma'(0.73)\,(0.5-0.67) = 0.22 \times (-0.17) = -0.04$$

$$\delta_1^{(1)} = \sigma'(-1.00)\big[(-0.04)(2)+(-0.04)(1)\big] = 0.20(-0.12) = -0.02$$
$$\delta_2^{(1)} = \sigma'(1.00)\big[(-0.04)(-1)+(-0.04)(2)\big] = 0.20(-0.04) = -0.01$$

#### Error Summary

| Sample    | $\delta_1^{(2)}$ | $\delta_2^{(2)}$ | $\delta_1^{(1)}$ | $\delta_2^{(1)}$ |
| --------- | ---------------- | ---------------- | ---------------- | ---------------- |
| $S^{(1)}$ | 0.10             | -0.15            | 0.01             | -0.04            |
| $S^{(2)}$ | -0.12            | 0.09             | -0.04            | 0.08             |
| $S^{(3)}$ | -0.04            | -0.04            | -0.02            | -0.01            |

---

### Step 3 — Per-Sample Weight Changes ($\eta = 1$)

Using $\Delta w_{ij}^{(l,s)} = \eta \cdot \delta_i^{(l)} \cdot o_j^{(l-1)}$

**Output layer** ($o_j^{(1)} \in \{h_1, h_2, 1\}$):

| Sample    | $\Delta w_{11}^{(2)}$ | $\Delta w_{12}^{(2)}$ | $\Delta w_{13}^{(2)}$ | $\Delta w_{21}^{(2)}$ | $\Delta w_{22}^{(2)}$ | $\Delta w_{23}^{(2)}$ |
| --------- | --------------------- | --------------------- | --------------------- | --------------------- | --------------------- | --------------------- |
| $S^{(1)}$ | $0.10(0.12)=0.012$    | $0.10(0.88)=0.088$    | $0.10(1)=0.10$        | $-0.15(0.12)=-0.018$  | $-0.15(0.88)=-0.132$  | $-0.15(1)=-0.15$      |
| $S^{(2)}$ | $-0.12(0.50)=-0.06$   | $-0.12(0.50)=-0.06$   | $-0.12(1)=-0.12$      | $0.09(0.50)=0.045$    | $0.09(0.50)=0.045$    | $0.09(1)=0.09$        |
| $S^{(3)}$ | $-0.04(0.27)=-0.0108$ | $-0.04(0.73)=-0.0292$ | $-0.04(1)=-0.04$      | $-0.04(0.27)=-0.0108$ | $-0.04(0.73)=-0.0292$ | $-0.04(1)=-0.04$      |

**Hidden layer** ($o_j^{(0)} \in \{x_1, x_2, 1\}$):

| Sample    | $\Delta w_{11}^{(1)}$ | $\Delta w_{12}^{(1)}$ | $\Delta w_{13}^{(1)}$ | $\Delta w_{21}^{(1)}$ | $\Delta w_{22}^{(1)}$ | $\Delta w_{23}^{(1)}$ |
| --------- | --------------------- | --------------------- | --------------------- | --------------------- | --------------------- | --------------------- |
| $S^{(1)}$ | $0.01(1)=0.01$        | $0.01(1)=0.01$        | $0.01(1)=0.01$        | $-0.04(1)=-0.04$      | $-0.04(1)=-0.04$      | $-0.04(1)=-0.04$      |
| $S^{(2)}$ | $-0.04(3)=-0.12$      | $-0.04(1)=-0.04$      | $-0.04(1)=-0.04$      | $0.08(3)=0.24$        | $0.08(1)=0.08$        | $0.08(1)=0.08$        |
| $S^{(3)}$ | $-0.02(0)=0$          | $-0.02(0)=0$          | $-0.02(1)=-0.02$      | $-0.01(0)=0$          | $-0.01(0)=0$          | $-0.01(1)=-0.01$      |

---

### Step 4 — Batch Weight Update (average over the 3 samples)

$$\Delta w_{ij}^{(l,\text{batch})} = \frac{1}{3}\sum_{s=1}^{3} \Delta w_{ij}^{(l,s)}$$

#### Output layer, $\Delta w^{(2,\text{batch})}$

$$\Delta w_{11}^{(2,\text{batch})} = \frac{0.012 -0.06 -0.0108}{3} = \frac{-0.0588}{3} \approx -0.02$$
$$\Delta w_{12}^{(2,\text{batch})} = \frac{0.088 -0.06 -0.0292}{3} = \frac{-0.0012}{3} \approx 0.00$$
$$\Delta w_{13}^{(2,\text{batch})} = \frac{0.10 -0.12 -0.04}{3} = \frac{-0.06}{3} = -0.02$$
$$\Delta w_{21}^{(2,\text{batch})} = \frac{-0.018+0.045-0.0108}{3} = \frac{0.0162}{3} \approx 0.01$$
$$\Delta w_{22}^{(2,\text{batch})} = \frac{-0.132+0.045-0.0292}{3} = \frac{-0.1162}{3} \approx -0.04$$
$$\Delta w_{23}^{(2,\text{batch})} = \frac{-0.15+0.09-0.04}{3} = \frac{-0.10}{3} \approx -0.03$$

#### Hidden layer, $\Delta w^{(1,\text{batch})}$

$$\Delta w_{11}^{(1,\text{batch})} = \frac{0.01-0.12+0}{3} = \frac{-0.11}{3} \approx -0.04$$
$$\Delta w_{12}^{(1,\text{batch})} = \frac{0.01-0.04+0}{3} = \frac{-0.03}{3} = -0.01$$
$$\Delta w_{13}^{(1,\text{batch})} = \frac{0.01-0.04-0.02}{3} = \frac{-0.05}{3} \approx -0.02$$
$$\Delta w_{21}^{(1,\text{batch})} = \frac{-0.04+0.24+0}{3} = \frac{0.20}{3} \approx 0.07$$
$$\Delta w_{22}^{(1,\text{batch})} = \frac{-0.04+0.08+0}{3} = \frac{0.04}{3} \approx 0.01$$
$$\Delta w_{23}^{(1,\text{batch})} = \frac{-0.04+0.08-0.01}{3} = \frac{0.03}{3} = 0.01$$

#### Final Batch Weight Adjustments

**Output layer (Layer 2):**

|            | into $y_1$                          | into $y_2$                          |
| ---------- | ----------------------------------- | ----------------------------------- |
| from $h_1$ | $\Delta w_{11}^{(2)} \approx -0.02$ | $\Delta w_{21}^{(2)} \approx 0.01$  |
| from $h_2$ | $\Delta w_{12}^{(2)} \approx 0.00$  | $\Delta w_{22}^{(2)} \approx -0.04$ |
| from $b_2$ | $\Delta w_{13}^{(2)} = -0.02$       | $\Delta w_{23}^{(2)} \approx -0.03$ |

**Hidden layer (Layer 1):**

|            | into $h_1$                          | into $h_2$                         |
| ---------- | ----------------------------------- | ---------------------------------- |
| from $x_1$ | $\Delta w_{11}^{(1)} \approx -0.04$ | $\Delta w_{21}^{(1)} \approx 0.07$ |
| from $x_2$ | $\Delta w_{12}^{(1)} = -0.01$       | $\Delta w_{22}^{(1)} \approx 0.01$ |
| from $b_1$ | $\Delta w_{13}^{(1)} \approx -0.02$ | $\Delta w_{23}^{(1)} = 0.01$       |

_(New weights, if applied, would simply be $w^{new} = w^{old} + \Delta w^{(\text{batch})}$, since $\eta=1$.)_

---

### Why average gradients over a batch instead of updating after every single sample?

Updating the weights after **every single data point** (pure online/stochastic learning) means each update is based on the error of just one example. Individual samples can be noisy, unusual, or even mislabeled outliers, so the resulting gradient direction can be very erratic — the weights would "zig-zag" and the loss could actually increase from one step to the next for the batch as a whole.

By **averaging the gradients across a batch** before updating:

- The update direction represents the _overall_ trend of the error across many examples, canceling out noise from individual samples.
- Training becomes **more stable** and less sensitive to any single outlier sample.
- It allows for a more efficient use of parallel hardware (GPUs), since the whole batch can be processed together in matrix form.
- It provides a better (lower-variance) estimate of the true gradient of the loss function over the full data distribution, which generally leads to smoother, more reliable convergence toward a good minimum.

The trade-off is that batch updates require more computation and memory per update step, and pure batch gradient descent (using the entire dataset every step) can be slow to converge for very large datasets — which is why in practice **mini-batch gradient descent** (a middle ground between single-sample and full-batch) is most commonly used.
